### Script to combine the systems of UKMO into one anomaly timeseries.  Assumes anomalies from lead dependent climatology for each system have already been calculated

In [1]:
import xarray as xr
import numpy as np
import pandas
import yaml
import os
import xesmf as xe

In [2]:
model='UKMO'
with open("../../../DATA_DOWNLOAD/system_year_info.yaml") as f:
    model_info = yaml.safe_load(f)
info = model_info[model]
systems = info['forecast'].keys()

#### Information e.g., variable to use, start and end years, start and end years of hindcast period

In [3]:
var = 'slp'
yhind1 = 1993 ; yhind2 = 2016
basepath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/"+model+"/nc/"+var+"/"
outpath="/glade/campaign/cgd/cas/islas/DATASETS/CDS/seasonal/combine_systems/"+model+"/"

# Outputting also to the same location where the other data are
outpath2="/glade/campaign/cgd/cas/islas/python_savs/rossbypalooza26/remove_lead_dependent_climo/"+model+"/"
os.makedirs(outpath, exist_ok=True)

#### Select the last system as the hindcast system

In [4]:
hindcast_sys = list(systems)[-1]

#### Read in the hindcast data and the forecast data.  Combined and check all years are there
#### Also regridding to a common 1 degree grid because the systems are not necessarily all on the same grid

In [5]:
grid_out = xr.Dataset({'lat':(['lat'], np.arange(-90,90,1))}, {'lon': (['lon'], np.arange(0,360,1))})

In [6]:
alldat_ALLM=[]
hindcast_ALLM = xr.open_dataset(basepath+str(hindcast_sys)+'/hindcast_anoms_ALLM_'+var+'.nc')
regridder = xe.Regridder(hindcast_ALLM, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                         filename='/glade/derecho/scratch/islas/temp/wgt.nc')
hindcast_ALLM_rg = regridder(hindcast_ALLM)
alldat_ALLM.append(hindcast_ALLM_rg)

alldat_INDM=[]
hindcast_INDM = xr.open_dataset(basepath+str(hindcast_sys)+'/hindcast_anoms_INDM_'+var+'.nc')
regridder = xe.Regridder(hindcast_INDM, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                         filename='/glade/derecho/scratch/islas/temp/wgt.nc')
hindcast_INDM_rg = regridder(hindcast_INDM)
alldat_INDM.append(hindcast_INDM_rg)


for isys in systems:
    print(isys)
    years = info['forecast'][isys]
    dat = xr.open_dataset(basepath+str(isys)+'/forecast_anoms_ALLM_'+var+'.nc')
    regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                             filename='/glade/derecho/scratch/islas/temp/wgt.nc')
    dat_rg = regridder(dat)
    alldat_ALLM.append(dat_rg.sel(year=years))

    dat = xr.open_dataset(basepath+str(isys)+'/forecast_anoms_INDM_'+var+'.nc')
    regridder = xe.Regridder(dat, grid_out, 'bilinear', periodic=True, reuse_weights=False,
                             filename='/glade/derecho/scratch/islas/temp/wgt.nc')
    dat_rg = regridder(dat)
    alldat_INDM.append(dat_rg.sel(year=years))


alldat_ALLM = xr.concat(alldat_ALLM, dim='year')
alldat_ALLM.to_netcdf(outpath+var+'_anoms_ALLM_'+str(int(np.min(alldat_ALLM.year)))+'_'+str(int(np.max(alldat_ALLM.year)))+'.nc')
alldat_ALLM.to_netcdf(outpath2+var+'_anoms_ALLM_'+str(int(np.min(alldat_ALLM.year)))+'_'+str(int(np.max(alldat_ALLM.year)))+'.nc')


alldat_INDM = xr.concat(alldat_INDM, dim='year')
alldat_INDM.to_netcdf(outpath+var+'_anoms_INDM_'+str(int(np.min(alldat_INDM.year)))+'_'+str(int(np.max(alldat_INDM.year)))+'.nc')
alldat_INDM.to_netcdf(outpath2+var+'_anoms_INDM_'+str(int(np.min(alldat_INDM.year)))+'_'+str(int(np.max(alldat_INDM.year)))+'.nc')

12
13
14
15
600
601
602
603
604


/glade/derecho/scratch/islas/tmp/ipykernel_78330/1067292588.py:32: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'member' ('member',) The recommendation is to set join explicitly for this case.
  alldat_ALLM = xr.concat(alldat_ALLM, dim='year')
